# What is new in ModelFlow

This notebook shows what has changed in ModelFlow since the ModelFlow book
was written, with an example of each change. They all use the Pakistan model
(`pak.pcim`) and a carbon tax scenario.

| Area | What is new | Section |
|---|---|---|
| Saving models | `.pcimz` fast format for large models, `.pcims` structure only files, file names not case sensitive | [Saving and loading models](#saving) |
| Reports | `model.table(scenarios=...)` makes tables of saved scenarios | [Tables with scenarios](#tables) |
| Solvers | next generation `_ng` solvers, including `newton_fbmin_ng`, which solves fast on the feedback set; `newton_implicit` fixed | [New solvers](#solvers) |
| Graphs | drawing without graphviz (`engine='svg'`), interactive `model.causality()` | [Graphs without graphviz](#graphs) |
| Notebooks | Jupyter Notebook 7 / JupyterLab 4, MyST and mermaid in markdown cells, `model.plain_html` for RISE slides | [Jupyter 7 and JupyterLab](#jupyter) |
| Input widgets | `%of` slider operators (percent of GDP), the same widget definitions in Shiny web apps | [Input widgets](#widgets) |
| Browser | ModelFlow runs in JupyterLite (Pyodide) without numba, graphviz or pyarrow | [Other changes](#other) |

Installation is described in the chapter *Installing ModelFlow*.

## Setup

The first cell installs ModelFlow when the notebook runs in Colab or in the
browser (JupyterLite). On your own PC it does nothing. It also downloads the
model file into the `data` folder if it is missing.

In [1]:
import sys
import importlib.util
from pathlib import Path

PAK_URL = 'https://raw.githubusercontent.com/IbHansen/modelflow-manual/main/model_repo/pak.pcim'

if 'google.colab' in str(get_ipython()):
    if importlib.util.find_spec("modelclass") is None:
        !apt-get -qq install -y graphviz
        !pip -qq install ModelFlowIb > /dev/null 2>&1

if sys.platform == 'emscripten':  # in the browser: JupyterLite
    import micropip
    await micropip.install('modelflowib>=2.86')
    from pyodide.http import pyfetch
    if not Path('data/pak.pcim').exists():
        Path('data').mkdir(exist_ok=True)
        response = await pyfetch(PAK_URL)
        Path('data/pak.pcim').write_bytes(await response.bytes())
elif not Path('data/pak.pcim').exists():
    import urllib.request
    Path('data').mkdir(exist_ok=True)
    urllib.request.urlretrieve(PAK_URL, 'data/pak.pcim')

In [2]:
from modelclass import model
model.widescreen()
model.scroll_off()

<IPython.core.display.Javascript object>

Load the model, run the baseline and two carbon tax scenarios. Each run is
kept under a name in `mpak.keep_solutions`, which the tables below use.

In [3]:
mpak, bline = model.modelload('data/pak.pcim', run=True, keep='Baseline')

CT30df = bline.upd("<2025 2050> PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER = 30")
CT60df = bline.upd("<2025 2050> PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER = 60")

_ = mpak(CT30df, 2020, 2050, keep='Carbon tax 30')
_ = mpak(CT60df, 2020, 2050, keep='Carbon tax 60')

list(mpak.keep_solutions)

Zipped file read:  data\pak.pcim
The feedback variables are not directly connected, so the feedback order is left as found


['Baseline', 'Carbon tax 30', 'Carbon tax 60']

(saving)=
## Saving and loading models: `.pcim`, `.pcimz` and `.pcims`

A model and its data are saved with `modeldump` and read back with
`modelload`. There are now three file formats:

| Suffix | Made with | Contains | Use it for |
|---|---|---|---|
| `.pcim` | `modeldump(file)` | equations, descriptions, data (json, gzipped) | the normal format, readable by all ModelFlow versions |
| `.pcimz` | `modeldump(file, large=True)` | the same, with the data in the fast Feather format (zip) | large models (CGE, 100,000+ variables), where `.pcim` is slow; needs `pyarrow` |
| `.pcims` | `modeldump(file, data=False)` | the model structure only, no data | sharing a model without its data, or storing the specification |

`keep=True` also saves the kept scenarios (`keep_solutions`).

In [4]:
Path('saved').mkdir(exist_ok=True)

mpak.modeldump('saved/pak_ct', keep=True)            # -> saved/pak_ct.pcim
mpak.modeldump('saved/pak_ct', keep=True, large=True) # -> saved/pak_ct.pcimz (needs pyarrow)
mpak.modeldump('saved/pak_structure', data=False)     # -> saved/pak_structure.pcims

for f in sorted(Path('saved').iterdir()):
    print(f'{f.name:25} {f.stat().st_size/1_000_000:8.2f} MB')

Model dumped (feather/zip): saved\pak_ct.pcimz
pak_ct.pcim                   3.82 MB
pak_ct.pcimz                  2.46 MB
pak_structure.pcims           0.03 MB


`compact=True` (only with `large=True`) stores the numbers as float32, which
halves the file. They are float64 again when the file is loaded.

`modelload` finds the format by itself. Without a suffix it tries `.pcimz`,
then `.pcim`, then `.pcims`:

In [5]:
mpak2, df2 = model.modelload('saved/pak_ct')   # finds saved/pak_ct.pcimz
list(mpak2.keep_solutions)

Feather/zip file read: saved\pak_ct.pcimz
The feedback variables are not directly connected, so the feedback order is left as found


['Baseline', 'Carbon tax 30', 'Carbon tax 60']

A `.pcims` file has no data, so it must be loaded with `run=False`, and the
dataframe returned is `None`. Give the model a dataframe yourself, for
instance one from another source with the same variables:

In [6]:
mstruct, nodata = model.modelload('saved/pak_structure.pcims', run=False)
print(f'{nodata=}, {len(mstruct.endogene)} endogenous variables')

res = mstruct(bline, 2020, 2050)     # solve with the baseline data from before
res.loc[2020:2025, 'PAKNYGDPMKTPKN']

Zipped file read:  saved\pak_structure.pcims
The feedback variables are not directly connected, so the feedback order is left as found
Structure only model loaded, there is no dataframe in pak_structure.pcims
nodata=None, 378 endogenous variables


2020    2.627394e+07
2021    2.651137e+07
2022    2.668514e+07
2023    2.696308e+07
2024    2.739320e+07
2025    2.796323e+07
Name: PAKNYGDPMKTPKN, dtype: float64

File names are no longer case sensitive when loading: `data/PAK.pcim` finds
`data/pak.pcim`. Windows never cared, but Linux, the Codespace and
JupyterLite do.

(tables)=
## Tables with scenarios

`model.table()` can now make tables from the kept scenarios, with the
`scenarios=` option, as `model.plot()` can.

- `scenarios='*'` takes all kept scenarios, in the order they were kept.
- Exact names are separated with `|`, for example `'Baseline|Carbon tax 60'`.
- Patterns with `*` or `?` are separated with spaces, for example `'Carbon*'`.
- For impact datatypes (`difpctlevel`, `diflevel`, `difgrowth`, ...), the
  **first** selected scenario is the reference.
- Without `scenarios=`, `table` uses basedf and lastdf as before.

Levels for all scenarios. Each scenario has a headline row above its rows:

In [7]:
tvars = 'PAKNYGDPMKTPKN PAKNECONPRVTKN PAKCCEMISCO2TKN'

mpak.table(tvars, datatype='level', scenarios='*', title='Levels in the scenarios',
           timeslice=[2025, 2030, 2040, 2050])

Impacts in percent of the baseline. The headline is *Carbon tax 30 vs Baseline*:

In [8]:
impacts = mpak.table(tvars, datatype='difpctlevel', scenarios='Baseline|Carbon tax 30|Carbon tax 60',
                     title='Percent difference from the baseline',
                     timeslice=[2025, 2030, 2040, 2050])
impacts

`scenario_layout='suffix'` gives one row per variable and scenario, labelled
*variable — scenario*. Transposed tables always use this layout.

In [9]:
mpak.table(tvars, datatype='difpctlevel', scenarios='Baseline|Carbon tax 30|Carbon tax 60',
           title='Percent difference from the baseline', scenario_layout='suffix',
           timeslice=[2025, 2030, 2040, 2050])

The table can also be shown as text and LaTeX, and its numbers are a dataframe:

In [10]:
impacts.show

Percent difference from the baseline
                             2025    2030    2040   2050
                              --- Impact in percent --- 
Carbon tax 30 vs Baseline
Real GDP                     0.36    0.18    0.25   0.35
HH. Cons Real               -0.87   -1.61   -0.68  -0.13
PAKCCEMISCO2TKN            -22.17  -18.55  -10.71  -5.90
Carbon tax 60 vs Baseline
Real GDP                     0.41    0.25    0.37   0.52
HH. Cons Real               -1.40   -2.48   -1.14  -0.29
PAKCCEMISCO2TKN            -31.11  -26.99  -16.67  -9.66


(solvers)=
## New solvers

A new family of solvers is in `modelsolver_ng.py`. They have the suffix
`_ng` (next generation) and are chosen with `solver=`. The legacy solvers are
still the default, so existing notebooks give the same results.

| Solver | Model type | What it does |
|---|---|---|
| `sim_ng`, `sim1d_ng`, `xgenr_ng` | normalized | the Gauss-Seidel and sweep solvers in the new framework |
| `newton_ng` | normalized, implicit and hybrid | Newton per period on the whole model, one solver for all equation types |
| `newton_fbmin_ng` | normalized, no leads | Newton only on the minimal feedback set (6 variables in PAK), the rest is solved exactly in topological order. The fastest solver on PAK |
| `newtonstack_ng` | all, with leads | stacked Newton over all periods; `split=1` leaves the parts outside the simultaneous core out of the Newton system |
| `newtonstack_fbmin_ng` | normalized, with leads | stacked Newton on the stacked feedback set only |

Also:

- `newton_implicit` now gives the right solution on normalized and hybrid
  models. Before, it could give wrong results.
- `stats=True` shows the time used, including the time for building the
  Jacobian.
- The options of a call are remembered for the next call. Use
  `reset_options=True` when switching solvers, so options from one solver
  don't leak into the next.

The chapter *Solving models in ModelFlow* (`solvers/solvers.ipynb`) describes
all the solvers and their options.

Below, the carbon tax scenario is solved with four solvers. The time and the
largest difference from `sim` are shown for each:

In [13]:
import time

results = {}
times = {}
for solver in ['sim', 'sim_ng', 'newton_ng', 'newton_fbmin_ng']:
    print(solver) 
    mpak(CT30df, 2020, 2050, solver=solver, reset_options=True, silent=1)   # warm up
    t0 = time.perf_counter()
    results[solver] = mpak(CT30df, 2020, 2050, solver=solver, silent=1)
    times[solver] = time.perf_counter() - t0

endo = sorted(mpak.endogene)
ref = results['sim'].loc[2020:2050, endo]
for solver, res in results.items():
    maxrel = ((res.loc[2020:2050, endo] - ref).abs() / ref.abs().clip(lower=1)).max().max()
    print(f'{solver:18} {times[solver]:7.3f} s   max rel. diff from sim {maxrel:.1e}')

sim
sim_ng
newton_ng
Houston, we have a problem!
This equation raises an error:

>>: FRML <DAMP,STOC> PAKNEGDIFPRVKN = (PAKNEGDIFPRVKN_A*PAKNEGDIKSTKKN(-1)+ (0.00212272413966296+0.970234989019907*(PAKNEGDIFPRVKN(-1)/PAKNEGDIKSTKKN(-2))+(1-0.970234989019907)*(((LOG(PAKNYGDPPOTLKN))-(LOG(PAKNYGDPPOTLKN(-1))))+PAKDEPR)+0.0525240494260597*((LOG(PAKNEKRTTOTLCN/PAKNYGDPFCSTXN))-(LOG(PAKNEKRTTOTLCN(-1)/PAKNYGDPFCSTXN(-1))))) *PAKNEGDIKSTKKN(-1)) * (1-PAKNEGDIFPRVKN_D)+ PAKNEGDIFPRVKN_X*PAKNEGDIFPRVKN_D  $
In: 2042 

Below you will find information which might help you identify the problem

>> Values of terms in used:
              term         value
    PAKNEGDIFPRVKN -9.677378e+04
  PAKNEGDIFPRVKN_A -2.890421e-05
PAKNEGDIKSTKKN(-1)  7.026126e+07
PAKNEGDIFPRVKN(-1)  2.648515e+06
PAKNEGDIKSTKKN(-2)  6.829522e+07
    PAKNYGDPPOTLKN  4.569968e+07
PAKNYGDPPOTLKN(-1)  4.714470e+07
           PAKDEPR  5.000000e-02
    PAKNEKRTTOTLCN -1.810074e-02
    PAKNYGDPFCSTXN  6.187202e+00
PAKNEKRTTOTLCN(-1) 

ValueError: expected a positive input, got -0.0029255133744435394

The differences are within the convergence tolerance. Gauss-Seidel stops
when the relative changes are small, Newton when the residuals are small.

`stats=True` prints where the time goes:

In [14]:
_ = mpak(CT30df, 2020, 2050, solver='newton_fbmin_ng', stats=True)

Setup time (seconds)                 :           0.04
Feedback (Newton) variables          :              6
DAG core variables                   :             98
DAG sweeps (outer iterations)        :            194
Inner Newton steps (fb eqs only)     :            163
Simulation time (seconds)            :           0.09


With `ljit=1` the model is compiled with numba. The first call compiles
(about 10 seconds on PAK), and later calls are several times faster. It is
worth it when the model is solved many times. It is not available in the
browser.

```python
mpak(CT30df, 2020, 2050, solver='newton_fbmin_ng', ljit=1)
```

(graphs)=
## Graphs without graphviz

`model.draw` used to need the graphviz program. Now networkx can draw the
graphs too, which also works in the browser:

- `engine='svg'` draws an svg. It stays sharp when you zoom, and hovering
  over a box or an arrow shows its description and formula.
- `engine='mpl'` draws with matplotlib.
- `engine='dot'` (the default) uses graphviz. If graphviz fails, ModelFlow
  falls back to networkx.

In [15]:
mpak.draw('PAKNYGDPMKTPKN', up=1, down=0, engine='svg')

`model.causality(var)` is a new interactive causality graph. Click on a
variable to move the graph to it, and press *Back* to go back. It works in
JupyterLab, Notebook 7, VS Code and JupyterLite, with neither graphviz nor
Dash. It needs `anywidget`.

In [16]:
mpak.causality('PAKNYGDPMKTPKN', up=1, down=1)

ImportError: The causality viewer needs anywidget, install it with:
    conda install -c conda-forge anywidget
or  pip install anywidget

(jupyter)=
## Jupyter Notebook 7 and JupyterLab

ModelFlow now works in Jupyter Notebook 7, JupyterLab 4, JupyterLite and VS
Code. Notebook 7 is built on JupyterLab, and the classic notebook 6 is no
longer developed. What has changed for ModelFlow users:

- `model.widescreen()` and `model.scroll_off()` work in Notebook 7,
  JupyterLab and JupyterLite. `scroll_off` shows the whole output of a cell
  instead of a scroll box.
- `model.modelflow_auto()` no longer runs the notebook when it is opened.
  That only worked in classic notebook 6. Now it only makes the notebook
  wider.
- `model.display_toc()` lists the notebooks in a folder as links. They now
  open in JupyterLab on Windows too. The root folder name is not shown as a
  header any more; use `showroot=True` to show it.
- Tab completion (`mpak.<Tab>`) now lists all methods, including the ones
  from the mixin classes.
- `jupyterlab-myst` is installed with ModelFlow, so markdown cells can use
  MyST like a Jupyter Book: note boxes, and diagrams with mermaid. The next
  cell is an example.

:::{note}
This box is a MyST directive. It renders in JupyterLab and Notebook 7 with
`jupyterlab-myst`, and in the Jupyter Book.
:::

```{mermaid}
flowchart LR
    A[upd: scenario dataframe] --> B[mpak: solve]
    B --> C[keep_solutions]
    C --> D[table / plot]
```

In [17]:
mpak.display_toc(text='**Notebooks in this folder**')

**Notebooks in this folder**

whats_new,What is new in ModelFlow


### Plain html for RISE slides

RISE shows notebooks as slides, but it can't display the ipywidgets Tab and
Accordion containers. With `model.plain_html = True`, the result viewer
(`mpak['...']`) and `rplot` use plain html tabs and collapsible sections
instead. These display anywhere html is shown.

In [18]:
model.plain_html = True
mpak['PAKNYGDPMKTPKN PAKCCEMISCO2TKN']

In [19]:
model.plain_html = False   # back to ipywidgets

(widgets)=
## Input widgets

- Slider operators `%of` and `+%of` (and `%of_impulse`, `+%of_impulse`, for
  the first period only). The slider value is a percent of a divisor
  variable, typically GDP. The variable is set to that amount, or the amount
  is added to it.
- `modelinput_shiny.make_app` builds a Shiny web app from the same widget
  definitions as the notebook widgets. It also runs in the browser with
  Shinylive.
- The input widgets now start quickly for models with many variables.

`Pakistan/interactive.ipynb` has a full example. Here is a slider for
government transfers in percent of GDP:

In [20]:
from modelwidget_input import make_widget, updatewidget

gov_slidedef = ['slide', {
    'heading': 'Government expenditure, % of GDP',
    'content': {
        'Transfers to households': {'var': 'PAKGGEXPTRNSCN_X', 'value': 0.0, 'min': -2.0, 'max': 2.0,
                                    'step': 0.1, 'op': '+%of', 'divisor': 'PAKNYGDPMKTPCN'},
    }
}]

ui = updatewidget(mpak, make_widget(gov_slidedef), varpat='PAKNYGDPMKTPKN PAKNYGDPMKTPCN')
ui

(other)=
## Other changes

**ModelFlow in the browser and the cloud**
- ModelFlow runs in JupyterLite (Pyodide). numba, graphviz, dash and pyarrow
  are not needed there; pip leaves them out by itself.
- A GitHub Codespace with ModelFlow installed.

**Model specification and estimation**
- Syntax errors show where in the model the error is.
- A clear error message when the model text has no FRML statements.
- A warning when there are several equations for the same variable.
- `equpdate` accepts a `Makemodel` instance instead of an equation.
- Estimation: t-values, a syntax check before estimating, a check for
  variables missing from the data, and a check of the sample for missing
  lags and leads.
- `Estimate_ols` handles equations that mix imposed and estimated
  coefficients.

**Data and patterns**
- `mfcalc` and the dekomposition work with pandas 3.
- A variable pattern starting with `@` is a regular expression (for names
  and descriptions); without `@` it is the usual wildcard pattern.
- `modelgrabgdx` loads GAMS gdx files into dataframes.

**Reports and display**
- Markdown and html reports show continuation lines correctly.
- A model without data can still show its structure.

The full list is in `changelog.md` in the ModelFlow repository.